# Mock an OpenAI Chat Completions Call

Use a cell-defined Hono route to serve a local, OpenAI-compatible mock response. This notebook sends no request to OpenAI and needs no API key.

Run the cells from top to bottom in a JavaScript kernel.

## Claim a prefix and define the mock endpoint

The prefix is unique to this example so it is less likely to conflict with routes claimed by another notebook.

In [ ]:
const routePrefix = '/api/openai-mock/';
claimPrefix(routePrefix);

app.post(`${routePrefix}v1/chat/completions`, async c => {
  const request = await c.req.json();
  const userMessage = request.messages?.find(message => message.role === 'user');
  const prompt = typeof userMessage?.content === 'string' ? userMessage.content : 'your message';
  const model = request.model ?? 'gpt-4o-mini';
  const created = Math.floor(Date.now() / 1000);
  const id = 'chatcmpl-mock-cell-routes';
  const assistantText = `This is a mocked response. I received: ${prompt}`;

  if (request.stream === true) {
    const textChunks = assistantText.match(/[\s\S]{1,12}/gu) ?? [assistantText];
    const makeChunk = choices => ({ id, object: 'chat.completion.chunk', created, model, choices });
    const frames = [
      makeChunk([{ index: 0, delta: { role: 'assistant' }, finish_reason: null }]),
      ...textChunks.map(content =>
        makeChunk([{ index: 0, delta: { content }, finish_reason: null }])
      ),
      makeChunk([{ index: 0, delta: {}, finish_reason: 'stop' }])
    ];
    const encoder = new TextEncoder();
    const body = new ReadableStream({
      start(controller) {
        for (const frame of frames) {
          controller.enqueue(encoder.encode(`data: ${JSON.stringify(frame)}\n\n`));
        }
        controller.enqueue(encoder.encode('data: [DONE]\n\n'));
        controller.close();
      }
    });

    return c.body(body, 200, {
      'content-type': 'text/event-stream; charset=utf-8',
      'cache-control': 'no-cache'
    });
  }

  return c.json({
    id,
    object: 'chat.completion',
    created,
    model,
    choices: [
      {
        index: 0,
        message: { role: 'assistant', content: assistantText },
        finish_reason: 'stop'
      }
    ],
    usage: { prompt_tokens: 0, completion_tokens: 0, total_tokens: 0 }
  });
});

Confirm that this kernel owns the route prefix before making the request.

In [ ]:
await claimedPrefixes()

## Make an OpenAI-compatible call

This uses the Chat Completions request and response shape. The request goes to the claimed same-origin route, so the kernel's Runtime serves it instead of a remote API.

In [ ]:
const response = await fetch('/api/openai-mock/v1/chat/completions', {
  method: 'POST',
  headers: { 'content-type': 'application/json' },
  body: JSON.stringify({
    model: 'gpt-4o-mini',
    messages: [{ role: 'user', content: 'Explain cell-defined routes in one sentence.' }]
  })
});

if (!response.ok) {
  throw new Error(`Mock OpenAI request failed: ${response.status} ${response.statusText}`);
}

const completion = await response.json();
console.log(completion.choices[0].message.content);
completion

## Make a streaming call

Set `stream: true` to receive Server-Sent Events in the OpenAI Chat Completions chunk format. The parser below reads event frames from the response body and appends each content delta, with a one-second pause between frames so the output is visibly paced.

In [ ]:
const streamResponse = await fetch('/api/openai-mock/v1/chat/completions', {
  method: 'POST',
  headers: { 'content-type': 'application/json' },
  body: JSON.stringify({
    model: 'gpt-4o-mini',
    messages: [{ role: 'user', content: 'What is streaming mode?' }],
    stream: true
  })
});

if (!streamResponse.ok) {
  throw new Error(`Mock OpenAI stream failed: ${streamResponse.status} ${streamResponse.statusText}`);
}

if (!streamResponse.body) {
  throw new Error('Mock OpenAI stream response has no body.');
}

const reader = streamResponse.body.getReader();
const decoder = new TextDecoder();
let buffer = '';
let streamedText = '';

while (true) {
  const { value, done } = await reader.read();
  buffer += decoder.decode(value, { stream: !done });

  let separator;
  while ((separator = buffer.indexOf('\n\n')) !== -1) {
    const event = buffer.slice(0, separator);
    buffer = buffer.slice(separator + 2);
    const data = event.split('\n').find(line => line.startsWith('data: '))?.slice(6);

    if (!data || data === '[DONE]') {
      continue;
    }

    const chunk = JSON.parse(data);
    const content = chunk.choices[0]?.delta?.content;
    if (content) {
      streamedText += content;
      console.log(content);
    }
    await new Promise(resolve => setTimeout(resolve, 1000));
  }

  if (done) {
    break;
  }
}

console.log(`Full response: ${streamedText}`);

The assistant text is generated by the Hono handler in this notebook's JavaScript kernel. An OpenAI-compatible client can be pointed at `/api/openai-mock/v1` to exercise the same local mock without contacting the real OpenAI API.

The route emits SSE frames for `stream: true`. The client pauses one second between parsed frames so the output is visibly paced. The current cell-route bridge buffers response bodies before forwarding them, so this demonstrates the streaming response format and parsing logic, but not incremental end-to-end delivery.